# Meta-Judge cho đánh giá sinh ngôn ngữ tiếng Việt

**Quy trình thực nghiệm.** Sử dụng nguồn, tham chiếu, bản dịch A/B và điểm
người chấm từ kết quả dịch đã lưu; xây dựng dữ liệu suy giảm ngữ nghĩa, đánh giá
metric và đối chiếu thứ hạng metric với đánh giá của con người.

## 1. Thiết lập thí nghiệm

Notebook tự tìm repo `work` khi chạy local, hoặc clone repo này khi chạy trên
Colab. Source, resource và notebook cùng một revision Git nên không cần upload
hay giải nén thủ công. Mọi kết quả nằm trong `ROOT/output/RUN_NAME`.

**Hai chế độ dữ liệu mặc định đều dùng lại artifact trong Git:**

| Tùy chọn | `False` (mặc định) | `True` |
|---|---|---|
| `REGENERATE_TRANSLATIONS` | Dùng `resources/source.csv` | Sinh lại hệ A bằng Gemini và hệ B bằng Google Translate |
| `REGENERATE_DAMAGE` | Dùng `resources/zero_shot.jsonl` và `few_shot.jsonl` | Sinh/resume damage mới vào output của `RUN_NAME` |

Khi bật sinh lại, các bước phía sau tự đọc artifact vừa sinh. Điểm human trong
`scores.xlsx` chỉ gắn với A/B đã chốt trong repo, nên tự tắt khi A/B được sinh
lại. Không có Gemini API nào được gọi nếu cả hai tùy chọn vẫn là `False`.

**Metric cache:** `RERUN_METRICS=False` và `USE_SOURCE_METRIC_CACHE=True`
đọc các archive trong `result/`, nạp score vào `metrics/fast/` và cho các
cell correlation/analysis chạy tiếp mà không gọi metric worker. Nếu archive
thiếu hoặc sai số dòng, notebook dừng rõ ràng; đặt `RERUN_METRICS=True` mới
cho phép chấm lại. `RERUN_METRICS=True` luôn ưu tiên hơn cache.

| Thư mục output | Nội dung |
|---|---|
| `translations/` | Bộ mẫu và checkpoint A/B khi chạy lại bước dịch |
| `data/` | Dữ liệu bản dịch đã kiểm tra và tập tham chiếu thí nghiệm |
| `generation/` | Zero-shot, few-shot, B1 và kiểm tra damage |
| `metrics/` | Điểm metric, tương quan và bảng kết quả |
| `analysis/` | B2, xuất điểm người chấm, phân tích lỗi và demo |
| `logs/` | Log riêng cho từng lượt chạy |

In [ ]:
import os
from pathlib import Path
import subprocess

# ── TÙY CHỌN NGƯỜI CHẠY ────────────────────────────────────────────────────
# False: dùng bản dịch A/B có sẵn trong Git. True: sinh/resume A/B mới.
REGENERATE_TRANSLATIONS = False

# False: dùng zero/few-shot damage có sẵn trong Git. True: sinh/resume damage mới.
REGENERATE_DAMAGE = False

# Key quota thường: toàn bộ mảng dùng chung rule cũ, tối đa một request mỗi 4.2s.
STANDARD_GEMINI_API_KEYS = []  # Chỉ điền key khi bật sinh lại.

# Key thuộc project có quota cao: chạy nhanh hơn theo cấu hình HIGH_QUOTA bên dưới.
# Không có API key nào thật sự "unlimited"; chỉ đưa key vào đây sau khi kiểm tra
# RPM/RPD của đúng model trên Google AI Studio. Nhiều key cùng một project vẫn
# dùng chung quota project. Không lặp key giữa hai mảng.
HIGH_QUOTA_GEMINI_API_KEYS = []  # Chỉ điền key khi bật sinh lại.

# True: cài dependency còn thiếu. Nên giữ True trên một runtime Colab mới.
INSTALL_DEPENDENCIES = True

# True: bật BERTScore, COMET và BLEURT; cần GPU để chạy nhanh.
RUN_HEAVY_METRICS = True

# True: chấm thử một config mỗi họ metric trên 24 hàng trước khi chạy full.
RUN_METRIC_SMOKE_TEST = True

# Đường chạy cũ khi không dùng source cache; RERUN_METRICS=True luôn chạy full.
RUN_FULL_METRICS = False

# Metric mode: mặc định đọc archive trong result/ và không chạy worker metric.
# Đặt RERUN_METRICS=True khi muốn chấm lại; cờ này có ưu tiên cao hơn cache.
RERUN_METRICS = False
USE_SOURCE_METRIC_CACHE = True
SOURCE_METRIC_CACHE_DIR_OVERRIDE = None  # Có thể trỏ tới thư mục result khác.

# True: đọc work/result/baseline.csv đã chấm sẵn; không chạy metric worker cho B1.
USE_BASELINE_CACHE = True
BASELINE_CSV_FILE_OVERRIDE = None  # Có thể trỏ tới baseline.csv khác.

# True: tính thêm BLEU/chrF sau khi tách từ bằng underthesea.
RUN_UNDERTHESEA = True

# ── HẰNG SỐ VÀ GIỚI HẠN THỰC NGHIỆM ─────────────────────────────────────────
# Đổi RUN_NAME khi thay model, prompt hoặc input; giữ nguyên để resume checkpoint.
RUN_NAME = "full-gemini-3-5-flash-lite-01"
GEMINI_MODEL = "gemini-3.5-flash-lite"
EXPECTED_ROWS = 300
LIMIT = None  # None dùng đủ 300 câu; số nguyên dương dùng cho pilot.
SEED = 42
MAX_OUTPUT_TOKENS = 400

# Pool thường giữ đúng rule cũ và dùng chung quota giữa các key trong mảng.
STANDARD_API_WORKERS = 1
STANDARD_API_MIN_INTERVAL_SECONDS = 4.2

# Mỗi key quota cao có pool riêng: mặc định 4 request đồng thời, cách nhau 0.25s.
# Giảm workers hoặc tăng interval nếu dashboard báo 429 do RPM.
HIGH_QUOTA_WORKERS_PER_KEY = 4
HIGH_QUOTA_MIN_INTERVAL_SECONDS = 0.25

CPU_WORKERS = 2
GPU_BATCH_SIZE = 16
HF_TOKEN = ""
LOCAL_SOURCE_VI = None
LOCAL_SOURCE_ZH = None
MANUAL_AUDIT_FILE = None
DEMO_SENTENCE = "Việt Nam đang đẩy mạnh chuyển đổi số trong giáo dục đại học."

# Repo mặc định chứa source/resource. Có thể override bằng biến môi trường.
WORK_REPO_URL = os.environ.get(
    "META_JUDGE_WORK_REPO",
    "https://github.com/thanhnghi-do-2k3/llm-as-judge.git",
)
WORK_REPO_BRANCH = "main"


def normalize_key_array(name, values):
    if not isinstance(values, (list, tuple)) or any(
        not isinstance(value, str) for value in values
    ):
        raise ValueError(f"{name} phải là mảng các chuỗi.")
    return list(dict.fromkeys(value.strip() for value in values if value.strip()))


STANDARD_GEMINI_API_KEYS = normalize_key_array(
    "STANDARD_GEMINI_API_KEYS", STANDARD_GEMINI_API_KEYS
)
HIGH_QUOTA_GEMINI_API_KEYS = normalize_key_array(
    "HIGH_QUOTA_GEMINI_API_KEYS", HIGH_QUOTA_GEMINI_API_KEYS
)
overlapping_keys = set(STANDARD_GEMINI_API_KEYS) & set(HIGH_QUOTA_GEMINI_API_KEYS)
if overlapping_keys:
    raise ValueError("Một Gemini key không được nằm trong cả hai mảng.")
ALL_GEMINI_API_KEYS = STANDARD_GEMINI_API_KEYS + HIGH_QUOTA_GEMINI_API_KEYS

if (REGENERATE_TRANSLATIONS or REGENERATE_DAMAGE) and not ALL_GEMINI_API_KEYS:
    raise ValueError(
        "Bật sinh lại nhưng chưa điền STANDARD_GEMINI_API_KEYS hoặc "
        "HIGH_QUOTA_GEMINI_API_KEYS."
    )


def is_work_repo(path):
    path = Path(path)
    return (
        (path / "resources" / "source.csv").is_file()
        and (path / "resources" / "scores.xlsx").is_file()
        and (path / "meta-judge" / "src" / "vn_meta_judge").is_dir()
    )


cwd = Path.cwd().resolve()
local_candidates = [cwd, cwd / "work"]
ROOT = next((path for path in local_candidates if is_work_repo(path)), None)

if ROOT is None:
    clone_root = Path("/content/llm-as-judge").resolve()
    if not is_work_repo(clone_root):
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                WORK_REPO_BRANCH,
                WORK_REPO_URL,
                str(clone_root),
            ],
            check=True,
        )
    ROOT = clone_root

ROOT = Path(ROOT).resolve()
RESOURCES_DIR = ROOT / "resources"
CODE_DIR = ROOT / "meta-judge"
SOURCE_METRIC_CACHE_DIR = (
    Path(SOURCE_METRIC_CACHE_DIR_OVERRIDE).expanduser().resolve()
    if SOURCE_METRIC_CACHE_DIR_OVERRIDE
    else ROOT / "result"
)
BASELINE_CSV_FILE = (
    Path(BASELINE_CSV_FILE_OVERRIDE).expanduser().resolve()
    if BASELINE_CSV_FILE_OVERRIDE
    else ROOT / "result" / "baseline.csv"
)
METRICS_FROM_SOURCE_CACHE = USE_SOURCE_METRIC_CACHE and not RERUN_METRICS
EFFECTIVE_RUN_HEAVY_METRICS = RUN_HEAVY_METRICS and not METRICS_FROM_SOURCE_CACHE
if RERUN_METRICS and USE_SOURCE_METRIC_CACHE:
    print("RERUN_METRICS=True: bỏ qua source metric cache và chạy lại metric.")
if (REGENERATE_TRANSLATIONS or REGENERATE_DAMAGE) and not ALL_GEMINI_API_KEYS:
    raise ValueError(
        "Bật sinh lại nhưng chưa điền STANDARD_GEMINI_API_KEYS hoặc "
        "HIGH_QUOTA_GEMINI_API_KEYS."
    )
if METRICS_FROM_SOURCE_CACHE and (REGENERATE_TRANSLATIONS or REGENERATE_DAMAGE):
    raise ValueError(
        "Không thể dùng metric cache với dữ liệu translation/damage vừa sinh; "
        "đặt RERUN_METRICS=True hoặc tắt các cờ REGENERATE."
    )

REPO_TRANSLATIONS_FILE = RESOURCES_DIR / "source.csv"
REPO_SCORES_FILE = RESOURCES_DIR / "scores.xlsx"
REPO_DAMAGE_FILES = {
    "zero_shot": RESOURCES_DIR / "zero_shot.jsonl",
    "few_shot": RESOURCES_DIR / "few_shot.jsonl",
}

# Các biến ACTIVE luôn trỏ tới artifact mà pipeline phía dưới phải sử dụng.
ACTIVE_TRANSLATIONS_FILE = REPO_TRANSLATIONS_FILE
ACTIVE_SOURCE_FILE = None  # source.csv đã chứa nguồn và tham chiếu.
ACTIVE_SCORES_FILE = REPO_SCORES_FILE

### Môi trường thực thi

Giữ nguyên scientific stack NumPy/Pandas/SciPy có sẵn của runtime. Khi dùng
metric cache, notebook không cài hoặc nạp BERTScore, COMET và BLEURT. Chỉ khi
`RERUN_METRICS=True` mới cô lập các package metric nặng theo từng thư mục.
Probe import dừng sớm nếu cache cài dở; không tạo virtualenv và không thay
PyTorch của Colab.

In [2]:
import hashlib
import json
import os
import shutil
import subprocess
import sys

ROOT = Path(ROOT).expanduser().resolve()
CODE_DIR = Path(CODE_DIR).expanduser().resolve()

if not (CODE_DIR / "src" / "vn_meta_judge" / "notebook_workflow.py").is_file():
    raise FileNotFoundError(f"Source Git chưa đầy đủ: {CODE_DIR}")
required_inputs = []
if not REGENERATE_TRANSLATIONS:
    required_inputs.extend([REPO_TRANSLATIONS_FILE, REPO_SCORES_FILE])
if not REGENERATE_DAMAGE:
    required_inputs.extend(REPO_DAMAGE_FILES.values())
for required_input in required_inputs:
    if not Path(required_input).is_file():
        raise FileNotFoundError(f"Thiếu resource trong Git: {required_input}")

if sys.version_info >= (3, 13) and EFFECTIVE_RUN_HEAVY_METRICS:
    raise RuntimeError("Metric nặng cần Python 3.12 trở xuống.")

os.environ["HF_HOME"] = str(ROOT / "cache" / "huggingface")
os.environ["NLTK_DATA"] = str(ROOT / "cache" / "nltk")
os.environ["TORCH_HOME"] = str(ROOT / "cache" / "torch")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MPLBACKEND"] = "Agg"

METRIC_PYTHON = Path(sys.executable).resolve()
HEAVY_WORKER_PATHS = {}


def run_checked(command, label, env=None):
    result = subprocess.run(
        command,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.returncode != 0:
        tail = "\n".join(result.stdout.splitlines()[-40:])
        raise RuntimeError(f"{label} thất bại:\n{tail}")
    return result.stdout


SCIENTIFIC_DISTRIBUTIONS = ("numpy", "pandas", "scipy")
VERSION_SCRIPT = """
import importlib.metadata as metadata
import json
import sys

versions = {}
for name in sys.argv[1:]:
    try:
        versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        pass
print(json.dumps(versions, sort_keys=True))
"""


def installed_versions(names):
    output = run_checked(
        [sys.executable, "-c", VERSION_SCRIPT, *names],
        "Đọc phiên bản scientific stack",
    )
    return json.loads(output.strip())


def ensure_kernel_matches_disk(installed):
    mismatches = []
    for name in SCIENTIFIC_DISTRIBUTIONS:
        module = sys.modules.get(name)
        loaded = getattr(module, "__version__", None)
        current = installed.get(name)
        if loaded and current and loaded != current:
            mismatches.append(f"{name}: RAM={loaded}, disk={current}")
    if mismatches:
        raise RuntimeError(
            "Runtime đã nạp scientific stack khác phiên bản đang có trên disk: "
            + "; ".join(mismatches)
            + ". Hãy chọn Runtime → Disconnect and delete runtime, kết nối lại "
            "rồi Run all notebook mới nhất."
        )


scientific_versions = installed_versions(SCIENTIFIC_DISTRIBUTIONS)
ensure_kernel_matches_disk(scientific_versions)

if INSTALL_DEPENDENCIES:
    constraint_path = ROOT / "cache" / "runtime-scientific-constraints.txt"
    constraint_path.parent.mkdir(parents=True, exist_ok=True)
    constraint_path.write_text(
        "".join(
            f"{name}=={version}\n"
            for name, version in sorted(scientific_versions.items())
        ),
        encoding="utf-8",
    )
    install_command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade-strategy",
        "only-if-needed",
    ]
    if scientific_versions:
        install_command.extend(["--constraint", str(constraint_path)])
    install_command.extend(
        ["-r", str(CODE_DIR / "requirements-notebook.txt")]
    )
    run_checked(install_command, "Cài dependency notebook")

    versions_after_install = installed_versions(SCIENTIFIC_DISTRIBUTIONS)
    changed_versions = {
        name: (version, versions_after_install.get(name))
        for name, version in scientific_versions.items()
        if versions_after_install.get(name) != version
    }
    if changed_versions:
        raise RuntimeError(
            "Pip đã thay scientific stack dù có constraint: "
            f"{changed_versions}. Hãy xóa runtime và báo lại log cài đặt."
        )
    ensure_kernel_matches_disk(versions_after_install)

    run_checked(
        [
            sys.executable,
            "-c",
            "import numpy, pandas, scipy; "
            "print(numpy.__version__, pandas.__version__, scipy.__version__)",
        ],
        "Kiểm tra ABI NumPy/Pandas/SciPy",
    )

    if EFFECTIVE_RUN_HEAVY_METRICS:
        family_requirements = {
            "BERTScore": "requirements-metric-bertscore.txt",
            "COMET": "requirements-metric-comet.txt",
            "BLEURT": "requirements-metric-bleurt.txt",
        }
        family_probes = {
            "BERTScore": "import bert_score, transformers",
            "COMET": "import lightning_utilities, pytorch_lightning, comet",
            "BLEURT": "import bleurt, tf_slim, sentencepiece, tensorflow",
        }
        for family, requirement in family_requirements.items():
            requirement_path = CODE_DIR / requirement
            package_dir = ROOT / "cache" / "metric-packages" / family.lower()
            fingerprint = hashlib.sha256(
                requirement_path.read_bytes()
                + f"{sys.version_info.major}.{sys.version_info.minor}".encode()
            ).hexdigest()[:16]
            ready_marker = package_dir / f".ready-{fingerprint}"

            probe_env = os.environ.copy()
            probe_env["PYTHONPATH"] = os.pathsep.join(
                [str(package_dir), str(CODE_DIR / "src")]
            )
            probe = subprocess.run(
                [sys.executable, "-c", family_probes[family]],
                env=probe_env,
                text=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
            )
            if not ready_marker.is_file() or probe.returncode != 0:
                # Cache cài dở được bỏ riêng theo family, không ảnh hưởng model cache.
                if package_dir.is_dir():
                    shutil.rmtree(package_dir)
                package_dir.mkdir(parents=True, exist_ok=True)
                print(f"Cài package tối thiểu cho metric: {family}")
                run_checked(
                    [
                        sys.executable,
                        "-m",
                        "pip",
                        "install",
                        "-q",
                        "--upgrade",
                        "--no-deps",
                        "--target",
                        str(package_dir),
                        "-r",
                        str(requirement_path),
                    ],
                    f"Cài dependency {family}",
                )
                probe_env["PYTHONPATH"] = os.pathsep.join(
                    [str(package_dir), str(CODE_DIR / "src")]
                )
                run_checked(
                    [sys.executable, "-c", family_probes[family]],
                    f"Kiểm tra dependency {family}",
                    env=probe_env,
                )
                ready_marker.write_text("ready\n", encoding="utf-8")
            HEAVY_WORKER_PATHS[family] = package_dir

sys.path.insert(0, str(CODE_DIR / "src"))

import pandas as pd

from IPython.display import display
from vn_meta_judge.notebook_workflow import NotebookExperiment, parallel_jobs

experiment = NotebookExperiment(
    root=ROOT,
    code_dir=CODE_DIR,
    run_name=RUN_NAME,
    model=GEMINI_MODEL,
    limit=LIMIT,
    seed=SEED,
    api_keys=STANDARD_GEMINI_API_KEYS,
    high_quota_api_keys=HIGH_QUOTA_GEMINI_API_KEYS,
    api_workers=STANDARD_API_WORKERS,
    api_min_interval_seconds=STANDARD_API_MIN_INTERVAL_SECONDS,
    high_quota_workers_per_key=HIGH_QUOTA_WORKERS_PER_KEY,
    high_quota_min_interval_seconds=HIGH_QUOTA_MIN_INTERVAL_SECONDS,
    cpu_workers=CPU_WORKERS,
    worker_python=METRIC_PYTHON,
    heavy_worker_pythonpaths=HEAVY_WORKER_PATHS,
    heavy=EFFECTIVE_RUN_HEAVY_METRICS,
    underthesea=RUN_UNDERTHESEA,
    max_tokens=MAX_OUTPUT_TOKENS,
)

print("Output:", experiment.output)
print("Git workspace:", ROOT)
print(
    "Gemini keys:",
    f"standard={len(STANDARD_GEMINI_API_KEYS)},",
    f"high_quota={len(HIGH_QUOTA_GEMINI_API_KEYS)},",
    f"request_workers={experiment.api_workers}",
)
print(
    "Chế độ dữ liệu:",
    f"translations={'regenerate' if REGENERATE_TRANSLATIONS else 'repo'},",
    f"damage={'regenerate' if REGENERATE_DAMAGE else 'repo'}",
)
print("Chế độ metric:", "rerun" if RERUN_METRICS else "source-cache", "| cache=", SOURCE_METRIC_CACHE_DIR)
print("Baseline cache:", BASELINE_CSV_FILE if USE_BASELINE_CACHE else "disabled")

Cài package tối thiểu cho metric: BERTScore
Cài package tối thiểu cho metric: COMET
Cài package tối thiểu cho metric: BLEURT
Output: /content/llm-as-judge/output/full-gemini-3-5-flash-lite-01
Git workspace: /content/llm-as-judge
Gemini keys: standard=13, high_quota=1, request_workers=5
Chế độ dữ liệu: translations=repo, damage=repo


In [ ]:
# Kaggle có thể clone bản repo chưa chứa notebook_cache.py.
# Cell này tự bổ sung đúng cache loader trước khi cell generate import nó.
from pathlib import Path

_CACHE_MODULE_SOURCE = '"""Reuse archived metric results from the repository in the main notebook.\n\nThe cache loader is deliberately read-only with respect to the source archive:\nit validates and extracts the archive into the current run directory, then\npoints ``NotebookExperiment`` at the extracted scored JSONL files.  Metric\nworkers are not started in this mode.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport shutil\nimport csv\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\n\ndef _safe_extract(archive: Path, destination: Path) -> None:\n    destination = destination.resolve()\n    with ZipFile(archive) as bundle:\n        for member in bundle.infolist():\n            target = (destination / member.filename).resolve()\n            if not target.is_relative_to(destination):\n                raise ValueError(f"Unsafe archive member: {member.filename}")\n        bundle.extractall(destination)\n\n\ndef _json(path: Path) -> dict:\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef _line_count(path: Path) -> int:\n    with path.open(encoding="utf-8") as handle:\n        return sum(1 for line in handle if line.strip())\n\n\ndef load_source_metric_cache(\n    experiment,\n    cache_dir: Path,\n    *,\n    branches=("zero_shot", "few_shot"),\n    tokenizations=("syllable", "underthesea"),\n) -> dict:\n    """Load archived results and attach them to ``experiment``.\n\n    The source archives must contain scored JSONL and metric score JSON for\n    every requested branch/tokenization.  A missing or malformed cache raises\n    an explicit error so a normal ``Run All`` cannot silently recreate scores.\n    Set the notebook\'s rerun flag when a fresh metric run is intended.\n    """\n\n    cache_dir = Path(cache_dir).expanduser().resolve()\n    if not cache_dir.is_dir():\n        raise FileNotFoundError(f"Không tìm thấy thư mục metric cache: {cache_dir}")\n\n    archive_paths = {}\n    for branch in branches:\n        candidates = sorted(cache_dir.glob(f"metric-results-{branch}-*.zip"))\n        if not candidates:\n            raise FileNotFoundError(\n                f"Không tìm thấy archive metric cho {branch} trong {cache_dir}"\n            )\n        archive_paths[branch] = candidates[-1]\n\n    extracted_root = experiment.output / "source-metric-cache"\n    extracted_root.mkdir(parents=True, exist_ok=True)\n    result_root = experiment.metrics_dir / "fast" / "results"\n    result_root.mkdir(parents=True, exist_ok=True)\n\n    # The human score file is shared by both branch archives.  Prefer the\n    # first branch and verify that both archives agree on its row count.\n    human_paths = []\n    report = {"status": "cached", "source_dir": str(cache_dir), "branches": {}}\n    for branch, archive in archive_paths.items():\n        digest = hashlib.sha256(archive.read_bytes()).hexdigest()\n        branch_root = extracted_root / branch\n        marker = branch_root / ".archive-sha256"\n        if not marker.is_file() or marker.read_text(encoding="utf-8").strip() != digest:\n            if branch_root.exists():\n                shutil.rmtree(branch_root)\n            branch_root.mkdir(parents=True, exist_ok=True)\n            _safe_extract(archive, branch_root)\n            marker.write_text(digest + "\\n", encoding="utf-8")\n\n        branch_result = {"archive": str(archive), "tokenizations": {}}\n        for tokenization in tokenizations:\n            scored = branch_root / "results" / f"scored_{branch}_{tokenization}.jsonl"\n            scores = branch_root / "results" / f"scores_{branch}_{tokenization}.json"\n            human = branch_root / "results" / f"scored_human_{tokenization}.jsonl"\n            human_scores = branch_root / "results" / f"scores_human_{tokenization}.json"\n            if (\n                not scored.is_file()\n                or not scores.is_file()\n                or not human.is_file()\n                or not human_scores.is_file()\n            ):\n                raise FileNotFoundError(\n                    f"Cache thiếu output {branch}/{tokenization}: {scored}, {scores}, "\n                    f"{human}, hoặc {human_scores}"\n                )\n            payload = _json(scores)\n            metric_count = len(payload.get("scores", {}))\n            row_count = _line_count(scored)\n            if metric_count == 0 or row_count == 0:\n                raise ValueError(f"Cache rỗng: {scores}")\n            for values in payload["scores"].values():\n                if len(values) != row_count:\n                    raise ValueError(\n                        f"Số điểm không khớp số dòng trong cache: {scores}"\n                    )\n            human_row_count = _line_count(human)\n            human_payload = _json(human_scores)\n            if not human_payload.get("scores") or any(\n                len(values) != human_row_count\n                for values in human_payload["scores"].values()\n            ):\n                raise ValueError(\n                    f"Số điểm human không khớp số dòng trong cache: {human_scores}"\n                )\n            destination = result_root / f"scores_{branch}_{tokenization}.json"\n            shutil.copy2(scores, destination)\n            shutil.copy2(\n                human_scores,\n                result_root / f"scores_human_{tokenization}.json",\n            )\n            # ``correlate`` validates score lengths against these paths.\n            experiment.datasets[branch] = scored\n            human_paths.append(human)\n            branch_result["tokenizations"][tokenization] = {\n                "rows": row_count,\n                "metrics": metric_count,\n                "scores": str(destination),\n            }\n\n        human_rows = _line_count(human_paths[-1])\n        expected_human_rows = (\n            _line_count(experiment.human_file)\n            if getattr(experiment, "human_file", None)\n            else len(experiment.human_rows)\n        )\n        if human_rows != expected_human_rows:\n            raise ValueError(\n                f"Cache human có {human_rows} dòng, input hiện tại có "\n                f"{expected_human_rows} dòng: {human_paths[-1]}"\n            )\n        report["branches"][branch] = branch_result\n\n    # The archived human score vectors are stored in both archives.  Use the\n    # first one as the dataset used by ``correlate`` and keep the clean human\n    # file from ``experiment.prepare`` for the human correlation calculation.\n    experiment.datasets["human"] = human_paths[0]\n    report["human_rows"] = _line_count(human_paths[0])\n    report["status"] = "ready"\n    return report\n\n\ndef load_baseline_csv_cache(\n    experiment,\n    csv_path: Path,\n    *,\n    branch: str = "rule_based",\n    tokenization: str = "syllable",\n) -> dict:\n    """Attach a pre-scored rule-based baseline CSV without running workers."""\n\n    csv_path = Path(csv_path).expanduser().resolve()\n    if not csv_path.is_file():\n        raise FileNotFoundError(f"Không tìm thấy baseline cache: {csv_path}")\n    with csv_path.open(encoding="utf-8-sig", newline="") as handle:\n        rows = list(csv.DictReader(handle))\n    required = {\n        "id",\n        "source_zh",\n        "reference_vi",\n        "prediction_vi",\n        "damage_level",\n        "prompt_type",\n        "model_name",\n        "backend",\n        "temperature",\n        "operation",\n        "status",\n    }\n    metric_columns = [name for name in (rows[0] if rows else {}) if name.startswith("metric__")]\n    if not rows or not required.issubset(rows[0]) or not metric_columns:\n        raise ValueError(f"Baseline cache thiếu schema hoặc metric columns: {csv_path}")\n    keys = {(row["id"], int(row["damage_level"])) for row in rows}\n    if len(rows) != 1800 or len(keys) != 1800:\n        raise ValueError(f"Baseline cache phải có 1.800 khóa (id, level): {csv_path}")\n    per_id = {}\n    for row in rows:\n        per_id.setdefault(row["id"], []).append(int(row["damage_level"]))\n    if len(per_id) != 300 or any(sorted(levels) != list(range(6)) for levels in per_id.values()):\n        raise ValueError(f"Baseline cache phải có 300 câu × level 0..5: {csv_path}")\n\n    extracted = experiment.output / "source-metric-cache" / branch\n    extracted.mkdir(parents=True, exist_ok=True)\n    dataset_path = extracted / f"scored_{branch}_{tokenization}.jsonl"\n    score_path = experiment.metrics_dir / "fast" / "results" / f"scores_{branch}_{tokenization}.json"\n    score_path.parent.mkdir(parents=True, exist_ok=True)\n    normalized = []\n    scores = {name.removeprefix("metric__"): [] for name in metric_columns}\n    for row in rows:\n        item = {\n            "id": row["id"],\n            "source_zh": row["source_zh"],\n            "reference_vi": row["reference_vi"],\n            "prediction_vi": row["prediction_vi"],\n            "damage_level": int(row["damage_level"]),\n            "prompt_type": row["prompt_type"],\n            "model_name": row["model_name"],\n            "backend": row["backend"],\n            "temperature": float(row["temperature"]),\n            "operation": row["operation"],\n            "status": row["status"],\n        }\n        normalized.append(item)\n        for name in metric_columns:\n            scores[name.removeprefix("metric__")].append(float(row[name]))\n    dataset_path.write_text(\n        "".join(json.dumps(item, ensure_ascii=False) + "\\n" for item in normalized),\n        encoding="utf-8",\n    )\n    score_path.write_text(\n        json.dumps({"scores": scores}, ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    experiment.datasets[branch] = dataset_path\n    return {\n        "status": "ready",\n        "branch": branch,\n        "tokenization": tokenization,\n        "rows": len(normalized),\n        "metrics": len(scores),\n        "source": str(csv_path),\n        "scores": str(score_path),\n    }\n'
_CACHE_MODULE_PATH = CODE_DIR / "src" / "vn_meta_judge" / "notebook_cache.py"
if not _CACHE_MODULE_PATH.is_file():
    _CACHE_MODULE_PATH.parent.mkdir(parents=True, exist_ok=True)
    _CACHE_MODULE_PATH.write_text(_CACHE_MODULE_SOURCE, encoding="utf-8")
    print("Đã bổ sung cache loader vào runtime.")


## 2. Chuẩn bị dữ liệu và sinh bản dịch

Kế thừa notebook gốc: VLSP2022 vi–zh, chọn 300 câu với seed 42, Gemini
3.5 Flash Lite cho hệ A, Google Translate cho hệ B và ghép kết quả theo STT.
Mặc định dùng dữ liệu đã lưu. Bật `REGENERATE_TRANSLATIONS` để chạy lại các cell
lấy mẫu/sinh bản dịch; checkpoint giữ nguyên các câu đã dịch thành công.

In [3]:
if REGENERATE_TRANSLATIONS:
    if LOCAL_SOURCE_VI and LOCAL_SOURCE_ZH:
        vi_path, zh_path = Path(LOCAL_SOURCE_VI), Path(LOCAL_SOURCE_ZH)
    else:
        from concurrent.futures import ThreadPoolExecutor
        from huggingface_hub import hf_hub_download

        def download_source(language):
            return hf_hub_download(
                repo_id="VLSP2023-MT/ViBidirectionMT-Eval",
                filename=f"VLSP2022/Test/public test/test.vi-zh.2022.{language}",
                repo_type="dataset",
                token=HF_TOKEN or None,
            )

        with ThreadPoolExecutor(max_workers=2) as pool:
            vi_path, zh_path = pool.map(download_source, ["vi", "zh"])
else:
    print("Dùng dữ liệu dịch đã lưu.")

Dùng dữ liệu dịch đã lưu.


In [4]:
if REGENERATE_TRANSLATIONS:
    import random

    with open(vi_path, encoding="utf-8") as handle:
        vi_lines = [line.strip() for line in handle if line.strip()]
    with open(zh_path, encoding="utf-8") as handle:
        zh_lines = [line.strip() for line in handle if line.strip()]

    assert len(vi_lines) == len(zh_lines), "Lệch số dòng nguồn và tham chiếu."
    idx = sorted(random.Random(SEED).sample(range(len(vi_lines)), EXPECTED_ROWS))

    df_source = pd.DataFrame(
        [
            {
                "STT": stt,
                "ID_cau_VLSP": f"vi-zh-2022-test-{i + 1:04d}",
                "Cau_nguon_ZH": zh_lines[i],
                "Cau_tham_chieu_VI": vi_lines[i],
            }
            for stt, i in enumerate(idx, start=1)
        ]
    )
    translation_dir = experiment.output / "translations"
    translation_dir.mkdir(parents=True, exist_ok=True)
    df_source.to_csv(
        translation_dir / "300_sample.csv", index=False, encoding="utf-8-sig"
    )
    display(df_source.head())

### Sinh hệ A/B và lưu checkpoint

Giữ system prompt và temperature = 0 của notebook gốc. Hai hệ dịch cùng
một câu độc lập; ghi checkpoint và ghép STT sau khi nhận kết quả. Dùng lại
file kết quả để giữ chính xác các bản dịch của lượt chạy đã được chấm điểm.

In [5]:
if REGENERATE_TRANSLATIONS:
    from vn_meta_judge.translation_workflow import translate_dataset

    translated = translate_dataset(
        df_source,
        translation_dir,
        api_keys=ALL_GEMINI_API_KEYS,
        model=GEMINI_MODEL,
        request_caller=experiment.gemini_caller,
        source_workers=experiment.api_workers,
        target_workers=2,
    )
    ACTIVE_TRANSLATIONS_FILE = translation_dir / "translations_reusable.csv"
    ACTIVE_SOURCE_FILE = None
    # Điểm cũ thuộc hai bản dịch trong Git, không được gán cho A/B vừa sinh lại.
    ACTIVE_SCORES_FILE = None
    display(translated[["STT", "Ban_dich_He_A", "Ban_dich_He_B"]].head())
else:
    print("Dùng bản dịch A/B và điểm human đã chốt trong Git.")

Dùng bản dịch A/B và điểm human đã chốt trong Git.


### Đọc dữ liệu dịch đã lưu — cell debug độc lập

Cell này nhận XLSX có sheet `Cham_diem` hoặc CSV chứa nguồn, tham chiếu và
bản dịch A/B. Kiểm tra STT, số dòng, trường rỗng và sự khớp với bảng điểm.
Sau setup, bắt đầu từ đây để chạy sinh damage, metric và phân tích trên dữ liệu đã có.

In [6]:
translations = experiment.load_translations(
    ACTIVE_TRANSLATIONS_FILE,
    source_file=ACTIVE_SOURCE_FILE,
    scores_file=ACTIVE_SCORES_FILE,
    expected_rows=EXPECTED_ROWS,
    translation_generation=(
        "regenerated_translations"
        if globals().get("REGENERATE_TRANSLATIONS", False)
        else "reused_translations"
    ),
)

display(
    translations[
        [
            "STT",
            "Nguon_ZH",
            "Tham_chieu_VI",
            "Ban_dich_He_A",
            "Ban_dich_He_B",
        ]
    ].head()
)

print(f"Đã đọc {len(translations)} câu và {2 * len(translations)} bản dịch A/B.")
same_count = translations["Ban_dich_He_A"].eq(translations["Ban_dich_He_B"]).sum()
print(f"Hai hệ dịch giống nhau: {same_count}/{len(translations)} câu.")

,STT,Nguon_ZH,Tham_chieu_VI,Ban_dich_He_A,Ban_dich_He_B
0,1,在遭受新冠肺炎疫情不小影响的背景下，2022年对越南恢复和发展经济具有重要意义。,Năm 2022 là năm có ý nghĩa quan trọng trong ph...,Trong bối cảnh chịu ảnh hưởng không nhỏ từ đại...,Trong bối cảnh ảnh hưởng nặng nề của dịch bệnh...
1,2,与新冠肺炎病毒共处不仅是医疗卫生问题，而且还涉及国家经济及社会所有活动的运作方式。,Sống chung với dịch COVID-19 không chỉ là vấn ...,Việc chung sống với virus SARS-CoV-2 không chỉ...,Sống chung với Covid-19 không chỉ là vấn đề y ...
2,3,为使经济尽可能早地恢复并可持续发展，同塔省国会代表黎明欢表示，应考虑经济自主性，制定政策提高...,"Để nền kinh tế phát triển ổn định, sớm phục hồ...",Để nền kinh tế phục hồi và phát triển bền vững...,Để nền kinh tế phục hồi và phát triển bền vững...
3,4,尽早恢复经济、与病毒安全共处,"Sớm phục hồi kinh tế, sống chung an toàn với d...","Sớm phục hồi kinh tế, chung sống an toàn với v...",Khôi phục nền kinh tế càng sớm càng tốt và chu...
4,5,此外，进出口还实现了正增长。,"Ngoài ra, một số tín hiệu về xuất nhập khẩu vẫ...","Ngoài ra, xuất nhập khẩu cũng đạt mức tăng trư...","Ngoài ra, xuất nhập khẩu cũng đạt mức tăng trư..."


Đã đọc 300 câu và 600 bản dịch A/B.
Hai hệ dịch giống nhau: 6/300 câu.


## 3. Chuẩn bị dữ liệu và khảo sát

Lưu input của lượt chạy, tách tập human có điểm hợp lệ và chọn tham chiếu
cho thí nghiệm. Các dòng có cờ lỗi vẫn được giữ trong bản xuất đầy đủ để phân tích.

In [7]:
input_summary = experiment.prepare()
display(pd.DataFrame([input_summary]))

lengths = pd.DataFrame(
    {
        "Source characters": translations["Nguon_ZH"].str.len(),
        "Reference syllables": translations["Tham_chieu_VI"].str.split().str.len(),
    }
)
display(lengths.describe().round(2))

,references,selected_references,human_all,human_clean,human_excluded,translation_generation,human_evaluation
0,300,300,600,582,18,reused_translations,available


,Source characters,Reference syllables
count,300.00,300.00
mean,42.19,35.49
std,24.19,18.68
min,4.00,4.00
25%,24.00,22.00
50%,37.00,33.00
75%,56.25,46.00
max,174.00,122.00


## 4. Sinh dữ liệu suy giảm ngữ nghĩa

Run All mặc định dùng hai file damage đã chốt trong Git, kiểm tra lại schema và
sự khớp reference rồi tạo B1 theo luật. Đặt `REGENERATE_DAMAGE = True` khi cần
sinh một run mới; checkpoint vẫn lưu theo câu, prompt và mức damage.
Cache metric nhẹ được chuẩn bị song song với phần kiểm tra dữ liệu.

In [8]:
if globals().get("METRICS_FROM_SOURCE_CACHE", False):
    from vn_meta_judge.notebook_cache import load_source_metric_cache

    cached_metrics = load_source_metric_cache(
        experiment,
        SOURCE_METRIC_CACHE_DIR,
        branches=("zero_shot", "few_shot"),
        tokenizations=("syllable", "underthesea"),
    )
    generation = {
        branch: {
            "status": "ready",
            "rows": details["tokenizations"]["syllable"]["rows"],
            "source": "metric-cache",
        }
        for branch, details in cached_metrics["branches"].items()
    }
    baseline_cache = {"status": "skipped", "reason": "disabled"}
    if USE_BASELINE_CACHE:
        from vn_meta_judge.notebook_cache import load_baseline_csv_cache

        baseline_cache = load_baseline_csv_cache(
            experiment,
            BASELINE_CSV_FILE,
            branch="rule_based",
            tokenization="syllable",
        )
        generation["rule_based"] = {
            "status": "ready",
            "rows": baseline_cache["rows"],
            "source": "baseline-csv-cache",
        }
    initial_jobs = {
        "damage": generation,
        "metric_cache": cached_metrics,
        "baseline_cache": baseline_cache,
    }
    experiment.state["generation"] = generation
    experiment.state["metric_cache"] = cached_metrics
    generation_summary_path = experiment.generation_dir / "generation_summary.json"
    generation_summary_path.parent.mkdir(parents=True, exist_ok=True)
    generation_summary_path.write_text(
        json.dumps(generation, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
else:
    if REGENERATE_DAMAGE:
        # Nếu RUN_NAME này trước đó chỉ copy artifact Git, bỏ đúng bản copy đó để
        # tương thích bản notebook cũ. Checkpoint đã sinh khác nội dung vẫn được resume.
        for branch, source_path in REPO_DAMAGE_FILES.items():
            target_path = experiment.generation_dir / f"{branch}.jsonl"
            if (
                target_path.is_file()
                and source_path.is_file()
                and hashlib.sha256(target_path.read_bytes()).digest()
                == hashlib.sha256(source_path.read_bytes()).digest()
            ):
                target_path.unlink()

    initial_jobs = parallel_jobs(
        {
            "damage": lambda: experiment.generate(
                regenerate_damage=REGENERATE_DAMAGE,
                damage_files=(None if REGENERATE_DAMAGE else REPO_DAMAGE_FILES),
            ),
            "metric_cache": experiment.prepare_metric_cache,
        },
        workers=2,
    )

    generation = initial_jobs["damage"]
display(
    pd.DataFrame(
        [
            {
                "branch": name,
                **{
                    k: result.get(k)
                    for k in ["status", "rows", "api_error_rows", "reason"]
                },
            }
            for name, result in generation.items()
            if isinstance(result, dict)
        ]
    )
)
print("Metric cache:", initial_jobs["metric_cache"]["status"])
print("Baseline cache:", initial_jobs.get("baseline_cache", {"status": "generated"})["status"])

expected_generation = (
    ["zero_shot", "few_shot"] + (["rule_based"] if USE_BASELINE_CACHE else [])
    if METRICS_FROM_SOURCE_CACHE
    else ["rule_based", "zero_shot", "few_shot"]
)
generation_not_ready = [
    name for name in expected_generation
    if generation.get(name, {}).get("status") != "ready"
]
if generation_not_ready:
    raise RuntimeError(
        "Generation chưa hoàn tất: "
        + ", ".join(generation_not_ready)
        + ". Kiểm tra resource hoặc giữ nguyên RUN_NAME để resume checkpoint."
    )

,branch,status,rows,api_error_rows,reason
0,rule_based,ready,1800,0,None
1,zero_shot,ready,1800,0,None
2,few_shot,ready,1800,0,None


Metric cache: ready


## 5. Chấm điểm và kiểm định metric

Human, B1, zero-shot và few-shot được ghép bằng offset có kiểm tra hash. Mỗi
cấu hình metric nặng chỉ nạp model một lần rồi score toàn bộ dataset; metric
nhẹ chạy song song theo family. Output sau đó được tách về đúng thứ tự nguồn.

In [9]:
# Smoke test dùng một câu mỗi nhánh và một config đại diện cho từng họ metric.
if globals().get("METRICS_FROM_SOURCE_CACHE", False):
    smoke_result = {"status": "cached", "summaries": {}}
    print("Bỏ smoke test: đang dùng metric cache từ source.")
elif RUN_METRIC_SMOKE_TEST:
    smoke_result = experiment.metric_smoke_test(gpu_batch_size=4)
    display(
        pd.DataFrame(
            [
                {
                    "tokenization": name,
                    "status": item.get("status"),
                    "metrics": item.get("completed_metrics"),
                    "rows": item.get("combined_rows"),
                }
                for name, item in smoke_result["summaries"].items()
            ]
        )
    )
    if smoke_result["status"] != "ready":
        failed_logs = [
            item.get("log")
            for item in smoke_result["tasks"].values()
            if item.get("status") != "ready"
        ]
        raise RuntimeError(f"Smoke test metric thất bại. Log: {failed_logs}")
else:
    print("Bỏ smoke test theo cấu hình.")

,tokenization,status,metrics,rows
0,syllable,ready,7,24
1,underthesea,ready,2,24


In [ ]:
if globals().get("METRICS_FROM_SOURCE_CACHE", False):
    metric_result = {
        "status": "ready",
        "mode": "source-cache",
        "summaries": cached_metrics["branches"],
    }
    experiment.state["fast_metrics"] = metric_result
    print("Dùng metric cache từ source; không chạy lại metric worker.")
elif RUN_FULL_METRICS or globals().get("RERUN_METRICS", False):
    metric_result = experiment.score_fast(gpu_batch_size=GPU_BATCH_SIZE)
    display(
        pd.DataFrame(
            [
                {
                    "tokenization": name,
                    "status": item.get("status"),
                    "metrics": item.get("completed_metrics"),
                    "rows": item.get("combined_rows"),
                }
                for name, item in metric_result["summaries"].items()
            ]
        )
    )
    if metric_result["status"] != "ready":
        failed_logs = [
            item.get("log")
            for item in metric_result["tasks"].values()
            if item.get("status") != "ready"
        ]
        raise RuntimeError(f"Full metric chưa hoàn tất. Log: {failed_logs}")
else:
    metric_result = {
        "status": "skipped",
        "reason": "RUN_FULL_METRICS=False",
    }
    print(
        "Chế độ smoke-only: đã bỏ qua chấm full. "
        "Đặt RUN_FULL_METRICS=True sau khi smoke test chạy ổn."
    )

FULL_METRICS_READY = metric_result.get("status") == "ready"

### Tương quan với human judgment và mức damage

Sau khi mọi lượt chấm điểm kết thúc, tính `r_hum = corr(metric, human)`,
`r_syn = corr(metric, −damage)` và `MC = corr(r_hum, r_syn)`.
Bảng ghi số metric chung; các cấu hình có tương quan không xác định được ghi riêng.

In [ ]:
if FULL_METRICS_READY:
    correlation_table = experiment.correlate()
    display(correlation_table.round(4))
else:
    correlation_table = pd.DataFrame()
    print("Bỏ qua correlation vì chưa chạy full metric.")

Bỏ qua correlation vì chưa chạy full metric.


## 6. Đối chiếu baseline và xuất điểm

B1 được đọc từ `result/baseline.csv` đã chấm đủ metric khi bật `USE_BASELINE_CACHE=True`; cell này không chạy lại metric worker. B2 tính lại meta-correlation từ artifact của tác giả.


In [ ]:
if FULL_METRICS_READY:
    analysis_results = experiment.analyze(manual_audit=MANUAL_AUDIT_FILE)
    display(
        pd.DataFrame(
            [
                {
                    "task": name,
                    "status": item["status"],
                    "rows": item.get("rows"),
                }
                for name, item in analysis_results.items()
            ]
        )
    )

    baseline_comparison = correlation_table[
        correlation_table["branch"] == "rule_based"
    ].copy()
    if baseline_comparison.empty:
        print("Không có baseline cache trong correlation table.")
    else:
        print("So sánh B1 từ baseline.csv:")
        display(baseline_comparison.round(4))
else:
    analysis_results = {}
    print("Bỏ qua baseline và xuất điểm vì chưa chạy full metric.")

Bỏ qua baseline và xuất điểm vì chưa chạy full metric.


## 7. Phân tích lỗi

Tổng hợp lỗi generation, metric và thay đổi điểm theo mức damage. Mẫu audit
được xuất để người chấm ghi `observed_level`; kết quả audit chỉ tính trên nhãn đã cung cấp.

In [ ]:
if FULL_METRICS_READY:
    diagnostics = experiment.metric_diagnostics()
    display(diagnostics.head(12))

    audit_sample = experiment.manual_audit_sample(count=30)
    print("Mẫu kiểm tra mức damage:", audit_sample)
    print("Báo cáo lỗi:", experiment.analysis_dir / "error_analysis.json")
else:
    diagnostics = pd.DataFrame()
    audit_sample = None
    print("Bỏ qua phân tích lỗi vì chưa chạy full metric.")

Bỏ qua phân tích lỗi vì chưa chạy full metric.


## 8. Demo trên câu mới

Áp dụng B1 cho một câu ngoài benchmark, so sánh điểm BLEU/chrF qua sáu mức
damage. Demo minh họa độ nhạy metric; kết quả Gemini được báo cáo ở thí nghiệm chính.

In [ ]:
if globals().get("METRICS_FROM_SOURCE_CACHE", False):
    demo_table = pd.DataFrame()
    experiment.state["demo"] = {"status": "skipped", "reason": "source-cache-no-demo-score"}
    print("Bỏ qua demo mới: source cache không có score cho câu ngoài benchmark.")
elif FULL_METRICS_READY:
    demo_table = experiment.demo(DEMO_SENTENCE)
    display(demo_table)

    import matplotlib.pyplot as plt

    metric_columns = [c for c in demo_table.columns if c not in {"level", "text"}]
    if metric_columns:
        fig, ax = plt.subplots(figsize=(9, 4))
        for column in metric_columns:
            ax.plot(
                demo_table["level"],
                demo_table[column],
                marker="o",
                label=column,
            )

        ax.set(xlabel="Damage level", ylabel="Metric score", xticks=range(6))
        ax.grid(alpha=0.2)
        ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
        fig.tight_layout()
        fig.savefig(
            experiment.analysis_dir / "demo.png",
            dpi=160,
            bbox_inches="tight",
        )
        display(fig)
        plt.close(fig)
else:
    demo_table = pd.DataFrame()
    print("Bỏ qua demo vì chưa chạy full metric.")

Bỏ qua demo vì chưa chạy full metric.


In [ ]:
if FULL_METRICS_READY:
    manifest = experiment.finalize()
    print("Trạng thái:", manifest["status"])
    print("Danh mục kết quả:", experiment.output / "manifest.json")
    display(pd.DataFrame(manifest["issues"]))
else:
    smoke_status = (
        smoke_result.get("status")
        if RUN_METRIC_SMOKE_TEST
        else "skipped"
    )
    manifest = {
        "status": "smoke-only",
        "smoke_status": smoke_status,
        "issues": [],
    }
    print("Trạng thái: smoke-only | metric smoke:", smoke_status)

Trạng thái: smoke-only | metric smoke: ready
